# 03. Попередня обробка даних (Preprocessing)

На даному етапі дані готуються до навчання моделей машинного навчання.

Основні завдання:

- завантаження датасету;
- вибір цільової змінної;
- відбір ознак;
- кодування категоріальних даних;
- масштабування числових ознак;
- розділення на навчальну, валідаційну та тестову вибірки;
- збереження підготовлених даних.

In [22]:
# ============================================================
# IMPORT LIBRARIES
# ============================================================

import os
import sys
import importlib
import warnings

import numpy as np
import pandas as pd

import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.preprocessing import OneHotEncoder

warnings.filterwarnings("ignore")

In [14]:
# ============================================================
# CONNECT GOOGLE DRIVE
# ============================================================

from google.colab import drive

drive.mount("/content/drive", force_remount=False)

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [15]:
# ============================================================
# LOAD PROJECT
# ============================================================

PROJECT_DIR = "/content/drive/MyDrive/FitnessML_Master"

if PROJECT_DIR not in sys.path:
    sys.path.insert(0, PROJECT_DIR)

importlib.invalidate_caches()

print("✓ Project connected")
print(PROJECT_DIR)

✓ Project connected
/content/drive/MyDrive/FitnessML_Master


In [16]:
# ============================================================
# IMPORT PROJECT MODULES
# ============================================================

import config
import utils

config = importlib.reload(config)
utils = importlib.reload(utils)

utils.section("Project modules")

print("✓ config.py loaded")
print("✓ utils.py loaded")


PROJECT MODULES
✓ config.py loaded
✓ utils.py loaded


In [17]:
# ============================================================
# LOAD DATASET
# ============================================================

utils.section("Dataset")

df = utils.load_dataset()

df["date"] = pd.to_datetime(df["date"])

utils.set_plot_style()

print("✓ Dataset loaded")
print(config.DATASET_NAME)


DATASET
✓ Dataset loaded
health_fitness_tracking_365days.csv


# Вибір цільової змінної

На даному етапі визначається змінна, яку модель машинного навчання буде прогнозувати.

Для дослідження обрано показник **спалених калорій (`calories_burned`)**, оскільки він є безперервною числовою величиною та добре підходить для задач регресії.

Також формується перелік ознак, які використовуватимуться під час навчання моделей.

In [18]:
# ============================================================
# TARGET VARIABLE
# ============================================================

utils.section("Target variable")

TARGET = "calories_burned"

FEATURES = [
    "age",
    "gender",
    "steps",
    "heart_rate_avg",
    "sleep_hours",
    "exercise_minutes",
    "stress_level",
    "weight_kg",
    "bmi",
]

print(f"Target variable : {TARGET}")
print(f"Features         : {len(FEATURES)}")


TARGET VARIABLE
Target variable : calories_burned
Features         : 9


# Формування матриці ознак

Після визначення цільової змінної формується матриця незалежних ознак (`X`) та вектор цільової змінної (`y`), які будуть використовуватися на наступних етапах побудови моделей.

In [19]:
# ============================================================
# FEATURE MATRIX
# ============================================================

utils.section("Feature matrix")

X = df[FEATURES].copy()

y = df[TARGET].copy()

print(f"X shape : {X.shape}")
print(f"y shape : {y.shape}")

display(X.head())
display(y.head())


FEATURE MATRIX
X shape : (365000, 9)
y shape : (365000,)


,age,gender,steps,heart_rate_avg,sleep_hours,exercise_minutes,stress_level,weight_kg,bmi
0,56,F,9341,62.029621,9.368819,0.623979,2,73.496429,22.471978
1,56,F,10873,59.062818,6.358311,109.208987,3,68.237867,22.569858
2,56,F,6638,58.494078,6.099619,3.083319,4,81.687890,17.595609
3,56,F,6062,56.546095,7.584023,22.023327,10,86.379884,20.154137
4,56,F,10399,59.507172,7.327957,76.483061,8,81.782982,32.624040


,calories_burned
0,2230.230419
1,1840.454777
2,2284.231946
3,1620.464266
4,2264.528312


Числові та категоріальні ознаки обробляються різними методами.

In [20]:
numeric_features = X.select_dtypes(
    include=np.number
).columns.tolist()

categorical_features = X.select_dtypes(
    exclude=np.number
).columns.tolist()

utils.section("Feature types")

print("Numeric:")
print(numeric_features)

print()

print("Categorical:")
print(categorical_features)


FEATURE TYPES
Numeric:
['age', 'steps', 'heart_rate_avg', 'sleep_hours', 'exercise_minutes', 'stress_level', 'weight_kg', 'bmi']

Categorical:
['gender']


# Аналіз цільової змінної

Перед навчанням моделей необхідно оцінити розподіл цільової змінної.

Це дозволяє виявити можливі перекоси, аномальні значення та визначити, чи потрібні додаткові перетворення даних.

In [21]:
# ============================================================
# TARGET DISTRIBUTION
# ============================================================

utils.section("Target distribution")

display(y.describe().to_frame(name="Value"))

fig = plt.figure(figsize=(10, 5))

plt.hist(
    y,
    bins=30,
    edgecolor="black"
)

plt.title("Розподіл цільової змінної")

plt.xlabel(utils.label(TARGET))

plt.ylabel("Кількість записів")

plt.tight_layout()

utils.save_figure(
    fig,
    "03_target_distribution.png"
)


TARGET DISTRIBUTION


,Value
count,365000.000000
mean,1999.942722
std,300.058450
min,505.956221
25%,1797.603287
50%,1999.980702
75%,2201.965489
max,3408.341803


✓ Figure saved -> /content/drive/MyDrive/FitnessML_Master/figures/03_target_distribution.png


# Розділення даних

Для забезпечення об'єктивної оцінки моделей набір даних розділяється на навчальну, валідаційну та тестову вибірки.

Навчальна вибірка використовується для навчання моделей, валідаційна — для налаштування параметрів, а тестова — для фінальної оцінки якості.

In [23]:
# ============================================================
# TRAIN / VALIDATION / TEST SPLIT
# ============================================================

from sklearn.model_selection import train_test_split

utils.section("Train / Validation / Test split")

X_train, X_temp, y_train, y_temp = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=config.RANDOM_STATE
)

X_valid, X_test, y_valid, y_test = train_test_split(
    X_temp,
    y_temp,
    test_size=0.50,
    random_state=config.RANDOM_STATE
)

print(f"Train      : {len(X_train):,}")
print(f"Validation : {len(X_valid):,}")
print(f"Test       : {len(X_test):,}")


TRAIN / VALIDATION / TEST SPLIT
Train      : 292,000
Validation : 36,500
Test       : 36,500


In [24]:
# ============================================================
# STANDARD SCALER
# ============================================================

from sklearn.preprocessing import StandardScaler

utils.section("Feature scaling")

scaler = StandardScaler()

X_train_scaled = X_train.copy()
X_valid_scaled = X_valid.copy()
X_test_scaled = X_test.copy()

X_train_scaled[numeric_features] = scaler.fit_transform(
    X_train[numeric_features]
)

X_valid_scaled[numeric_features] = scaler.transform(
    X_valid[numeric_features]
)

X_test_scaled[numeric_features] = scaler.transform(
    X_test[numeric_features]
)

print("✓ Numeric features scaled")


FEATURE SCALING
✓ Numeric features scaled


In [25]:
# ============================================================
# ONE-HOT ENCODING
# ============================================================

from sklearn.preprocessing import OneHotEncoder

utils.section("Categorical encoding")

encoder = OneHotEncoder(
    handle_unknown="ignore",
    sparse_output=False
)

encoded_train = encoder.fit_transform(
    X_train_scaled[categorical_features]
)

encoded_valid = encoder.transform(
    X_valid_scaled[categorical_features]
)

encoded_test = encoder.transform(
    X_test_scaled[categorical_features]
)

print("✓ Gender encoded")


CATEGORICAL ENCODING
✓ Gender encoded


# Формування фінального набору ознак

Після масштабування числових ознак та кодування категоріальних даних формується фінальна матриця ознак.

Саме ці дані будуть використовуватися на наступному етапі навчання моделей машинного навчання.

In [26]:
# ============================================================
# FINAL FEATURE MATRICES
# ============================================================

utils.section("Final feature matrices")

# Numeric features
X_train_num = X_train_scaled[numeric_features].reset_index(drop=True)
X_valid_num = X_valid_scaled[numeric_features].reset_index(drop=True)
X_test_num = X_test_scaled[numeric_features].reset_index(drop=True)

# Encoded categorical features
encoded_columns = encoder.get_feature_names_out(categorical_features)

X_train_cat = pd.DataFrame(
    encoded_train,
    columns=encoded_columns
)

X_valid_cat = pd.DataFrame(
    encoded_valid,
    columns=encoded_columns
)

X_test_cat = pd.DataFrame(
    encoded_test,
    columns=encoded_columns
)

# Final matrices
X_train_final = pd.concat(
    [X_train_num, X_train_cat],
    axis=1
)

X_valid_final = pd.concat(
    [X_valid_num, X_valid_cat],
    axis=1
)

X_test_final = pd.concat(
    [X_test_num, X_test_cat],
    axis=1
)

print(f"Train:      {X_train_final.shape}")
print(f"Validation: {X_valid_final.shape}")
print(f"Test:       {X_test_final.shape}")


FINAL FEATURE MATRICES
Train:      (292000, 10)
Validation: (36500, 10)
Test:       (36500, 10)


In [27]:
utils.section("Processed dataset preview")

display(X_train_final.head())
display(y_train.head())


PROCESSED DATASET PREVIEW


,age,steps,heart_rate_avg,sleep_hours,exercise_minutes,stress_level,weight_kg,bmi,gender_F,gender_M
0,0.461711,0.500689,-1.314821,-0.441690,-0.867710,-0.869975,0.640116,1.170658,0.0,1.0
1,0.012129,1.375226,0.918677,0.576938,0.336857,0.870290,-0.324355,-0.977127,1.0,0.0
2,1.192281,-0.444363,1.185487,-0.066516,7.922176,-0.521922,-0.134408,0.267066,0.0,1.0
3,-1.055627,0.865022,-1.270926,1.913041,0.429437,1.218343,-0.708921,0.975607,0.0,1.0
4,-0.999430,-0.286048,0.365836,0.127214,0.055461,-0.173869,1.003009,-0.053756,0.0,1.0


,calories_burned
240950,1545.129865
203652,1908.997963
78158,2090.141898
108001,2422.516634
230292,1841.918250


In [28]:
# ============================================================
# SAVE DATASETS
# ============================================================

utils.section("Save processed datasets")

utils.save_table(X_train_final, "03_X_train.csv")
utils.save_table(X_valid_final, "03_X_valid.csv")
utils.save_table(X_test_final, "03_X_test.csv")

utils.save_table(y_train.to_frame(name=TARGET), "03_y_train.csv")
utils.save_table(y_valid.to_frame(name=TARGET), "03_y_valid.csv")
utils.save_table(y_test.to_frame(name=TARGET), "03_y_test.csv")


SAVE PROCESSED DATASETS
✓ Table saved -> /content/drive/MyDrive/FitnessML_Master/tables/03_X_train.csv
✓ Table saved -> /content/drive/MyDrive/FitnessML_Master/tables/03_X_valid.csv
✓ Table saved -> /content/drive/MyDrive/FitnessML_Master/tables/03_X_test.csv
✓ Table saved -> /content/drive/MyDrive/FitnessML_Master/tables/03_y_train.csv
✓ Table saved -> /content/drive/MyDrive/FitnessML_Master/tables/03_y_valid.csv
✓ Table saved -> /content/drive/MyDrive/FitnessML_Master/tables/03_y_test.csv


In [29]:
import joblib

utils.section("Save preprocessing objects")

joblib.dump(
    scaler,
    config.MODELS_DIR / "03_scaler.pkl"
)

joblib.dump(
    encoder,
    config.MODELS_DIR / "03_encoder.pkl"
)

print("✓ Scaler saved")
print("✓ Encoder saved")


SAVE PREPROCESSING OBJECTS
✓ Scaler saved
✓ Encoder saved


In [30]:
report = f"""
# Data preprocessing

## Dataset

Train: {len(X_train_final):,}

Validation: {len(X_valid_final):,}

Test: {len(X_test_final):,}

Target:

{TARGET}

Numeric features:

{len(numeric_features)}

Categorical features:

{len(categorical_features)}
"""

utils.save_report(
    report,
    "03_preprocessing.md"
)

✓ Report saved -> /content/drive/MyDrive/FitnessML_Master/reports/03_preprocessing.md


In [31]:
utils.section("Preprocessing completed")


PREPROCESSING COMPLETED
